# Assignment 3 — Semantic Search RAG vs Moment RAG

**Selected video:** https://www.youtube.com/watch?v=U8YI5S_WAY4

## Setup

In [1]:
%pip install -q youtube-transcript-api openai faiss-cpu python-dotenv tiktoken numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY in .env"

In [3]:
import json
import numpy as np
import faiss
import tiktoken
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "openai/gpt-4o-mini"
VIDEO_ID    = "U8YI5S_WAY4"

## 1. Fetch transcript

In [4]:
ytt = YouTubeTranscriptApi()
fetched = ytt.fetch(VIDEO_ID, languages=["en"])
raw = [{"text": s.text, "start": s.start, "duration": s.duration} for s in fetched]
print(f"Total segments: {len(raw)}")
print(f"Example: {raw[0]}")

Total segments: 107
Example: {'text': 'So, what are the tools of the trade that', 'start': 0.0, 'duration': 4.28}


---
# Part 1 — Baseline Semantic Search RAG

Strategy: join all text, split into **fixed chunks (~300 tokens, 50-token overlap)**, timestamps ignored.

In [5]:
enc = tiktoken.get_encoding("cl100k_base")

def split_chunks(segments, chunk_tokens=300, overlap=50):
    full_text = " ".join(s["text"] for s in segments)
    tokens = enc.encode(full_text)
    chunks = []
    step = chunk_tokens - overlap
    for i in range(0, len(tokens), step):
        chunk_tok = tokens[i : i + chunk_tokens]
        chunks.append(enc.decode(chunk_tok))
    return chunks

chunks = split_chunks(raw)
print(f"Chunks generated: {len(chunks)}")

Chunks generated: 4


In [6]:
def embed(texts):
    """Embed a list of texts in batches of 100."""
    all_vecs = []
    for i in range(0, len(texts), 100):
        batch = texts[i : i + 100]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_vecs.extend([d.embedding for d in resp.data])
    return np.array(all_vecs, dtype="float32")

chunk_vecs = embed(chunks)
print(f"Embedding shape: {chunk_vecs.shape}")

Embedding shape: (4, 1536)


In [7]:
# FAISS index (L2-normalized = cosine similarity)
faiss.normalize_L2(chunk_vecs)
dim = chunk_vecs.shape[1]
baseline_index = faiss.IndexFlatIP(dim)  # inner product after normalization = cosine
baseline_index.add(chunk_vecs)
print(f"Vectors indexed: {baseline_index.ntotal}")

Vectors indexed: 4


In [8]:
def baseline_query(question, top_k=4):
    q_vec = embed([question])
    faiss.normalize_L2(q_vec)
    scores, idxs = baseline_index.search(q_vec, top_k)
    context = "\n\n".join(chunks[i] for i in idxs[0])

    prompt = f"""Use the context below to answer the question.

Context:
{context}

Question: {question}
Answer:"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

### Baseline RAG — Test

In [9]:
QUERIES = [
    "What is a transformer and how does attention work?",
    "How are LLMs trained and what data do they use?",
    "What are the security risks of LLMs like prompt injection?",
]

baseline_answers = {}
for q in QUERIES:
    ans = baseline_query(q)
    baseline_answers[q] = ans
    print(f"Q: {q}\nA: {ans}\n{'='*60}")

Q: What is a transformer and how does attention work?
A: A transformer is a type of neural network architecture that has become the foundation for many state-of-the-art models in natural language processing (NLP) and other fields. Introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, transformers are designed to handle sequential data, such as text, more effectively than previous architectures like recurrent neural networks (RNNs).

The key innovation of transformers is the attention mechanism, which allows the model to weigh the importance of different words in a sequence when making predictions. Here's how attention works:

1. **Input Representation**: Each word in the input sequence is converted into a vector representation, often using embeddings.

2. **Query, Key, and Value Vectors**: For each word, three vectors are created: a query vector, a key vector, and a value vector. These vectors are derived from the input embeddings through learned linear transfo

Q: How are LLMs trained and what data do they use?
A: Large Language Models (LLMs) are trained using a process called unsupervised learning, where they learn patterns in language from vast amounts of text data. This data typically includes books, articles, websites, and other written content available on the internet. The training involves feeding the model sequences of text and adjusting its parameters to predict the next word in a sentence, thereby learning grammar, facts, and some level of reasoning.

The data used for training LLMs is diverse and extensive, allowing the models to understand and generate human-like text across various topics and styles. However, the specific datasets and the exact methods used can vary between different models and organizations. It's important to note that LLMs do not have access to real-time data or personal information unless explicitly provided during interactions.


Q: What are the security risks of LLMs like prompt injection?
A: The context provided does not explicitly mention security risks associated with large language models (LLMs) like prompt injection. However, based on general knowledge, here are some potential security risks related to LLMs, including prompt injection:

1. **Prompt Injection**: This occurs when a user manipulates the input to the model in a way that causes it to execute unintended commands or produce harmful outputs. For example, an attacker might craft a prompt that leads the model to disclose sensitive information or perform actions that it shouldn't.

2. **Data Leakage**: LLMs trained on large datasets may inadvertently generate outputs that include sensitive or proprietary information from their training data. This can lead to unintentional data exposure.

3. **Misinformation Generation**: LLMs can produce plausible-sounding but false information, which can be exploited to spread misinformation or disinformation.

4. 

---
# Part 2 — Moment RAG

Key difference: each "moment" preserves the **timestamp** (start/end in seconds).
Consecutive segments are grouped into blocks of ~10 entries (or when a pause > 3s is detected),
producing semantically coherent segments **with precise video location**.

The final answer cites the exact timestamp — the user can jump directly to that moment.

In [10]:
def build_moments(segments, group_size=10, gap_threshold=3.0):
    """Group transcript segments into moments.

    A new moment starts when:
    - a silence gap > gap_threshold seconds is detected, OR
    - group_size consecutive segments have been accumulated.
    """
    moments = []
    current = []

    for i, seg in enumerate(segments):
        if current:
            prev = segments[i - 1]
            gap = seg["start"] - (prev["start"] + prev["duration"])
            if gap > gap_threshold or len(current) >= group_size:
                moments.append(current)
                current = []
        current.append(seg)

    if current:
        moments.append(current)

    result = []
    for m in moments:
        start_sec = m[0]["start"]
        end_sec   = m[-1]["start"] + m[-1]["duration"]
        text      = " ".join(s["text"] for s in m)
        result.append({
            "text":      text,
            "start_sec": start_sec,
            "end_sec":   end_sec,
            "timestamp": _fmt(start_sec),
        })
    return result

def _fmt(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

moments = build_moments(raw)
print(f"Moments generated: {len(moments)}")
print(f"Example: {moments[0]['timestamp']} → {moments[0]['text'][:100]}...")

Moments generated: 11
Example: 0:00 → So, what are the tools of the trade that we have when we want to work with elements? >> Okay. >> Yes...


In [11]:
moment_texts = [m["text"] for m in moments]
moment_vecs  = embed(moment_texts)
faiss.normalize_L2(moment_vecs)

moment_index = faiss.IndexFlatIP(moment_vecs.shape[1])
moment_index.add(moment_vecs)
print(f"Moments indexed: {moment_index.ntotal}")

Moments indexed: 11


In [12]:
def moment_query(question, top_k=4):
    q_vec = embed([question])
    faiss.normalize_L2(q_vec)
    scores, idxs = moment_index.search(q_vec, top_k)

    retrieved = [moments[i] for i in idxs[0]]

    # each context line includes the video timestamp
    context_parts = [
        f"[{m['timestamp']}] {m['text']}" for m in retrieved
    ]
    context = "\n\n".join(context_parts)

    prompt = f"""Use the context below to answer the question.
Each excerpt includes a timestamp in [M:SS] format indicating where in the video it appears.
Cite the relevant timestamps in your answer.

Context:
{context}

Question: {question}
Answer:"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content, retrieved

### Moment RAG — Test

In [13]:
moment_answers = {}
for q in QUERIES:
    ans, retrieved = moment_query(q)
    moment_answers[q] = {"answer": ans, "moments": retrieved}
    print(f"Q: {q}")
    print(f"Moments retrieved: {[m['timestamp'] for m in retrieved]}")
    print(f"A: {ans}\n{'='*60}")

Q: What is a transformer and how does attention work?
Moments retrieved: ['0:00', '2:52', '2:28', '2:05']
A: The provided context does not contain any information about transformers or how attention works. The excerpts focus on tools for working with elements, planning, and the use of AI agents in teaching. If you have a specific source or additional context regarding transformers and attention mechanisms, please provide that, and I would be happy to help!


Q: How are LLMs trained and what data do they use?
Moments retrieved: ['3:13', '2:52', '0:00', '2:05']
A: The provided context does not explicitly detail how LLMs (Large Language Models) are trained or the specific data they use. However, it does mention aspects of their functionality and interaction, such as their ability to think, respond continuously, and engage in back-and-forth dialogue, which are indicative of their training on diverse datasets to facilitate conversational abilities. For example, the excerpt at [2:52] highlights the dynamic nature of AI agents and their responses, while [2:05] emphasizes their constant availability and ability to engage in discussions. 

For a comprehensive understanding of how LLMs are trained and the data they utilize, additional information outside of the provided context would be necessary.


Q: What are the security risks of LLMs like prompt injection?
Moments retrieved: ['3:13', '2:05', '1:20', '2:28']
A: The provided context does not explicitly mention security risks of LLMs (Large Language Models) such as prompt injection. However, it does discuss the capabilities and functionalities of LLMs, including their ability to think and generate plans based on user prompts. 

For a comprehensive understanding of security risks like prompt injection, one would typically consider how malicious inputs can manipulate the model's responses or behavior. Prompt injection can lead to unintended outputs, potentially exposing sensitive information or causing the model to act in ways that are not aligned with user intentions.

If you need specific information about security risks, it may be beneficial to consult additional resources or literature focused on the security aspects of LLMs.


---
# Comparison: Baseline vs Moment RAG

In [14]:
print("SIDE-BY-SIDE COMPARISON\n" + "="*70)
for q in QUERIES:
    print(f"\nQUESTION: {q}")
    print("\n--- BASELINE RAG ---")
    print(baseline_answers[q])
    print("\n--- MOMENT RAG ---")
    print(moment_answers[q]["answer"])
    print(f"\nTimestamps cited: {[m['timestamp'] for m in moment_answers[q]['moments']]}")
    print("-"*70)

SIDE-BY-SIDE COMPARISON

QUESTION: What is a transformer and how does attention work?

--- BASELINE RAG ---
A transformer is a type of neural network architecture that has become the foundation for many state-of-the-art models in natural language processing (NLP) and other fields. Introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, transformers are designed to handle sequential data, such as text, more effectively than previous architectures like recurrent neural networks (RNNs).

The key innovation of transformers is the attention mechanism, which allows the model to weigh the importance of different words in a sequence when making predictions. Here's how attention works:

1. **Input Representation**: Each word in the input sequence is converted into a vector representation, often using embeddings.

2. **Query, Key, and Value Vectors**: For each word, three vectors are created: a query vector, a key vector, and a value vector. These vectors are derived from 

---
## Comparison Summary

| | Baseline RAG | Moment RAG |
|---|---|---|
| **Chunking** | Fixed (~300 tokens) | Temporal coherence (~10 segs or silence gap) |
| **Metadata** | Plain text only | Text + timestamp (start/end seconds) |
| **Timestamp citation** | None | Yes — e.g.  in the video |
| **Traceability** | None | High — user can seek to exact moment |
| **Fragmentation** | May cut mid-sentence | Respects natural speech pauses |
| **Cost** | Equal | Equal (same API, same model) |

### Limitations
- Long moments dilute embedding specificity
- No cross-encoder reranker (unlike the original Moment RAG)
- Text-only — no visual branch (CLIP over frames)

### Possible improvements
- Add reranker:  to reorder top-k results
- Use Whisper for videos without auto-captions
- Expand to multi-source (PDF + slides) following the ARGUS architecture